In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/home-credit-default-risk/application_train.csv')

print("Shape:", df.shape)
print("\nTarget variable distribution:")
print(df['TARGET'].value_counts())
print("\nDefault Rate:", round(df['TARGET'].mean() * 100, 2), "%")

Shape: (307511, 122)

Target variable distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default Rate: 8.07 %


In [3]:

missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

print(f"Total columns: {df.shape[1]}")
print(f"Columns with missing data: {len(missing)}")
print(f"\nTop 20 columns with most missing data:")
print(missing.head(20).round(2))

Total columns: 122
Columns with missing data: 67

Top 20 columns with most missing data:
COMMONAREA_MEDI             69.87
COMMONAREA_MODE             69.87
COMMONAREA_AVG              69.87
NONLIVINGAPARTMENTS_MODE    69.43
NONLIVINGAPARTMENTS_MEDI    69.43
NONLIVINGAPARTMENTS_AVG     69.43
FONDKAPREMONT_MODE          68.39
LIVINGAPARTMENTS_AVG        68.35
LIVINGAPARTMENTS_MEDI       68.35
LIVINGAPARTMENTS_MODE       68.35
FLOORSMIN_MEDI              67.85
FLOORSMIN_MODE              67.85
FLOORSMIN_AVG               67.85
YEARS_BUILD_MODE            66.50
YEARS_BUILD_MEDI            66.50
YEARS_BUILD_AVG             66.50
OWN_CAR_AGE                 65.99
LANDAREA_AVG                59.38
LANDAREA_MEDI               59.38
LANDAREA_MODE               59.38
dtype: float64


In [4]:

threshold = 40
cols_to_drop = missing[missing > threshold].index.tolist()

print(f"Columns being dropped (>{threshold}% missing): {len(cols_to_drop)}")
print(cols_to_drop)

# Drop them
df_clean = df.drop(columns=cols_to_drop)
print(f"\nShape after dropping: {df_clean.shape}")
print(f"Columns remaining: {df_clean.shape[1]}")

Columns being dropped (>40% missing): 49
['COMMONAREA_MEDI', 'COMMONAREA_MODE', 'COMMONAREA_AVG', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_MODE', 'FLOORSMIN_MEDI', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'OWN_CAR_AGE', 'LANDAREA_AVG', 'LANDAREA_MEDI', 'LANDAREA_MODE', 'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI', 'BASEMENTAREA_AVG', 'EXT_SOURCE_1', 'NONLIVINGAREA_MEDI', 'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MODE', 'ELEVATORS_MEDI', 'ELEVATORS_MODE', 'ELEVATORS_AVG', 'WALLSMATERIAL_MODE', 'APARTMENTS_AVG', 'APARTMENTS_MEDI', 'APARTMENTS_MODE', 'ENTRANCES_MODE', 'ENTRANCES_MEDI', 'ENTRANCES_AVG', 'LIVINGAREA_AVG', 'LIVINGAREA_MODE', 'LIVINGAREA_MEDI', 'HOUSETYPE_MODE', 'FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'FLOORSMAX_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BEGINEXPLUATATION_AVG

In [5]:
print(df_clean.dtypes.value_counts())
print()
print(df_clean.select_dtypes(include='object').columns.tolist())

int64      41
float64    20
object     12
Name: count, dtype: int64

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']


In [6]:
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()

cat_summary = pd.DataFrame({
    'unique_values': df_clean[cat_cols].nunique(),
    'missing_pct': df_clean[cat_cols].isnull().mean() * 100,
    'sample_values': [df_clean[col].value_counts().index[:3].tolist() for col in cat_cols]
})

print(cat_summary)

                            unique_values  missing_pct  \
NAME_CONTRACT_TYPE                      2     0.000000   
CODE_GENDER                             3     0.000000   
FLAG_OWN_CAR                            2     0.000000   
FLAG_OWN_REALTY                         2     0.000000   
NAME_TYPE_SUITE                         7     0.420148   
NAME_INCOME_TYPE                        8     0.000000   
NAME_EDUCATION_TYPE                     5     0.000000   
NAME_FAMILY_STATUS                      6     0.000000   
NAME_HOUSING_TYPE                       6     0.000000   
OCCUPATION_TYPE                        18    31.345545   
WEEKDAY_APPR_PROCESS_START              7     0.000000   
ORGANIZATION_TYPE                      58     0.000000   

                                                                sample_values  
NAME_CONTRACT_TYPE                              [Cash loans, Revolving loans]  
CODE_GENDER                                                       [F, M, XNA]  
FLAG_

In [7]:
df_clean['OCCUPATION_TYPE'] = df_clean['OCCUPATION_TYPE'].fillna('Unknown')

df_clean['NAME_TYPE_SUITE'] = df_clean['NAME_TYPE_SUITE'].fillna(df_clean['NAME_TYPE_SUITE'].mode()[0])

df_clean['CODE_GENDER'] = df_clean['CODE_GENDER'].replace('XNA', 'F')

df_clean = df_clean.drop(columns=['WEEKDAY_APPR_PROCESS_START'])

org_default_rate = df_clean.groupby('ORGANIZATION_TYPE')['TARGET'].mean().sort_values(ascending=False)
print("Default rate by Organization Type (top 10):")
print(org_default_rate.head(10).round(4))

Default rate by Organization Type (top 10):
ORGANIZATION_TYPE
Transport: type 3    0.1575
Industry: type 13    0.1343
Industry: type 8     0.1250
Restaurant           0.1171
Construction         0.1168
Cleaning             0.1115
Industry: type 1     0.1107
Industry: type 3     0.1062
Realtor              0.1061
Agriculture          0.1047
Name: TARGET, dtype: float64


In [8]:
high_risk_orgs = org_default_rate[org_default_rate > 0.10].index.tolist()
medium_risk_orgs = org_default_rate[(org_default_rate >= 0.07) & (org_default_rate <= 0.10)].index.tolist()

def categorize_org(org):
    if org in high_risk_orgs:
        return 'High Risk'
    elif org in medium_risk_orgs:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_clean['ORG_RISK_TIER'] = df_clean['ORGANIZATION_TYPE'].apply(categorize_org)

print(df_clean['ORG_RISK_TIER'].value_counts())
print()
print(df_clean.groupby('ORG_RISK_TIER')['TARGET'].mean().round(4))

ORG_RISK_TIER
Medium Risk    140904
Low Risk       106589
High Risk       60018
Name: count, dtype: int64

ORG_RISK_TIER
High Risk      0.1057
Low Risk       0.0576
Medium Risk    0.0876
Name: TARGET, dtype: float64


In [9]:
binary_map = {'Y': 1, 'N': 0}
df_clean['FLAG_OWN_CAR'] = df_clean['FLAG_OWN_CAR'].map(binary_map)
df_clean['FLAG_OWN_REALTY'] = df_clean['FLAG_OWN_REALTY'].map(binary_map)

df_clean = pd.get_dummies(df_clean, 
                          columns=['NAME_CONTRACT_TYPE', 'CODE_GENDER',
                                   'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
                                   'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
                                   'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
                                   'ORG_RISK_TIER'],
                          drop_first=True)

df_clean = df_clean.drop(columns=['ORGANIZATION_TYPE'])

numeric_missing = df_clean.isnull().mean() * 100
numeric_missing = numeric_missing[numeric_missing > 0].sort_values(ascending=False)

print(f"Remaining columns with missing values: {len(numeric_missing)}")
print(numeric_missing.round(2))

Remaining columns with missing values: 16
EXT_SOURCE_3                  19.83
AMT_REQ_CREDIT_BUREAU_MON     13.50
AMT_REQ_CREDIT_BUREAU_YEAR    13.50
AMT_REQ_CREDIT_BUREAU_QRT     13.50
AMT_REQ_CREDIT_BUREAU_WEEK    13.50
AMT_REQ_CREDIT_BUREAU_DAY     13.50
AMT_REQ_CREDIT_BUREAU_HOUR    13.50
OBS_60_CNT_SOCIAL_CIRCLE       0.33
DEF_30_CNT_SOCIAL_CIRCLE       0.33
OBS_30_CNT_SOCIAL_CIRCLE       0.33
DEF_60_CNT_SOCIAL_CIRCLE       0.33
EXT_SOURCE_2                   0.21
AMT_GOODS_PRICE                0.09
AMT_ANNUITY                    0.00
CNT_FAM_MEMBERS                0.00
DAYS_LAST_PHONE_CHANGE         0.00
dtype: float64


In [10]:
df_clean['EXT_SOURCE_3_MISSING'] = df_clean['EXT_SOURCE_3'].isnull().astype(int)

amt_req_cols = ['AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_YEAR',
                'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_WEEK',
                'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_HOUR']

df_clean[amt_req_cols] = df_clean[amt_req_cols].fillna(0)

remaining_cols = ['EXT_SOURCE_3', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
                  'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
                  'EXT_SOURCE_2', 'AMT_GOODS_PRICE', 'AMT_ANNUITY',
                  'CNT_FAM_MEMBERS', 'DAYS_LAST_PHONE_CHANGE']

for col in remaining_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Dataset shape:", df_clean.shape)

Missing values remaining: 0
Dataset shape: (307511, 113)


In [11]:
df_clean.to_csv('../data/processed/application_train_clean.csv', index=False)
print("Saved successfully")
print(f"Final shape: {df_clean.shape}")

Saved successfully
Final shape: (307511, 113)
